## *Limpando os dados Base_Suja_Data_Cleansing_BP_SP_v2.xlsx*

 ####  
1: Importação, Limpeza e Geração de CNPJs Válidos


In [10]:
import os
import random
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)


# Função geradora de CNPJs matematicamente válidos (Módulo 11 - Receita Federal)
def gerar_cnpj_valido(semente=None):
    if semente is not None:
        random.seed(semente)
    base = [random.randint(0, 9) for _ in range(8)] + [0, 0, 0, 1]

    # Cálculo do 1º dígito verificador
    pesos_1 = [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
    soma_1 = sum(d * p for d, p in zip(base, pesos_1))
    resto_1 = soma_1 % 11
    dv1 = 0 if resto_1 < 2 else 11 - resto_1
    base.append(dv1)

    # Cálculo do 2º dígito verificador
    pesos_2 = [6, 5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
    soma_2 = sum(d * p for d, p in zip(base, pesos_2))
    resto_2 = soma_2 % 11
    dv2 = 0 if resto_2 < 2 else 11 - resto_2
    base.append(dv2)

    return "".join(map(str, base))


# Ler a nova base v3 forçando dtype=str para preservar zeros à esquerda (016, 001, 0001)
base = pd.read_excel(
    "../BASE_DADOS_SUJA/base_fornecedores_ficticia-v3.xlsx", dtype=str
)
df = base.copy()

# Remover a coluna fantasma 'Unnamed: 0' e espaços extras nos nomes das colunas
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
df.columns = df.columns.str.strip()

# 1. Tirar espaços nas pontas e espaços duplos no meio de TODAS as colunas
for col in df.columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    )

# 2. Manter código do fornecedor no padrão de 3 dígitos ('016', '036', '001'...)
df["PARTNER (Codigo BP)"] = (
    df["PARTNER (Codigo BP)"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(3)
)

# 3. Padronizar AGRUPAMENTO para 4 dígitos padrão SAP ('001' ou '1' -> '0001')
df["AGRUPAMENTO"] = (
    df["AGRUPAMENTO"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(4)
)

# 4. Colunas que devem ficar 100% em MAIÚSCULO
cols_maiusculas = [
    "AGRUPAMENTO",
    "FUNCAO_1",
    "FUNCAO_2",
    "BU_SORT1 (Termo Pesquisa)",
    "TAXTYPE",
    "REGION (UF)",
    "COUNTRY",
    "LGORT (Deposito)",
    "WAERS (Moeda)",
]
for col in cols_maiusculas:
    if col in df.columns:
        df[col] = df[col].str.upper()

# 5. Colunas de texto em Título (Primeira letra maiúscula + preposições minúsculas)
preposicoes = [" Da ", " De ", " Do ", " Das ", " Dos ", " E "]
cols_titulo = ["NAME_ORG1 (Razao Social)", "STREET (Rua)", "CITY1 (Cidade SP)"]

for col in cols_titulo:
    df[col] = df[col].str.title()
    for prep in preposicoes:
        df[col] = df[col].str.replace(prep, prep.lower(), regex=False)

# 6. Limpar CNPJ original apenas para manter a identificação correta das duplicatas na Célula 4
df["TAXNUM (CNPJ)"] = (
    df["TAXNUM (CNPJ)"].str.replace(r"\D", "", regex=True).str.zfill(14)
)

display(df.head(41
                ))


,PARTNER (Codigo BP),AGRUPAMENTO,FUNCAO_1,FUNCAO_2,NAME_ORG1 (Razao Social),BU_SORT1 (Termo Pesquisa),TAXTYPE,TAXNUM (CNPJ),STREET (Rua),HOUSE_NUM1 (Numero),POST_CODE1 (CEP SP),CITY1 (Cidade SP),REGION (UF),COUNTRY,BUKRS (Empresa),EKORG (Org Compras),ZTERM (Cond Pagamento),AKONT (Conta Conciliacao),KOSTL (Centro de Custo),LGORT (Deposito),BANKL (Chave Banco),BANKN (Conta Bancaria),WAERS (Moeda)
0,016,0001,FLVN00,FLVN01,Metalúrgica Sol Nascente Ltda,SOL NASCEN,BR1,26819472000131,Rua Jurubatuba,845,09.725-000,Sao Bernardo do Campo,SP,BR,1410,2006,0001,12119000,1410,141A,341,11223-4,BRL
1,036,0001,FLVN00,FLVN01,Transportadora Meridional Ltda,MERIDIONAL,BR1,46819247000163,"Avenida Paulista, 1578",<NA>,01310-200,São Paulo,SP,BR,1410,2006,0001,12119000,1410,1,104,33210-8,BRL
2,010,0001,FLVN00,FLVN01,Distribuidora Aurora Eireli,AURORA,BR1,20846193000174,Rua Sete de Setembro,2890,07011-020,Guarulhos,SP,BR,1410,2006,1,12119000,2006,0001,237,99887-1,BRL
3,015,0001,FLVN00,FLVN01,Ferramentas Bela Vista Epp,BELA VISTA,BR1,25891437000128,"Rua Voluntários da Pátria, 3184",<NA>,02011000,Sao Paulo,SP,BR,1410,1410,0001,12119000,1410,1,237,99887-1,BRL
4,032,0001,FLVN00,FLVN01,Embalagens Cambuí Sul Ltda,CAMBUÍ SUL,BR1,42891634000182,Avenida Moraes Salles,1730,13.010-002,Campinas,SP,BR,1410,1410,0001,12119000,2006,141A,341,11223-4,BRL
5,025,0001,FLVN00,FLVN01,Plásticos Sorocabana Me,SOROCABANA,BR1,35841926000172,"Rua Quinze de Novembro, 2740",<NA>,18010-080,Sorocaba,SP,BR,1410,1410,1,12119000,1410,1,1,56789-0,BRL
6,003,0001,FLVN00,FLVN01,Química Serra Azul Epp,SERRA AZUL,BR1,13619482000153,Avenida Portugal,945,09040-000,Santo Andre,SP,BR,1410,2006,0001,12119000.0,2006,0001,341,11223-4,BRL
7,027,0001,FLVN00,FLVN01,Metalúrgica Paraíba Ltda.,PARAÍBA,BR1,37849152000168,"Avenida Nelson D'Ávila, 820",<NA>,12245030,São José dos Campos,SP,BR,1410,2006,0001,12119000,1410,141A,104,33210-8,BRL
8,006,0001,FLVN00,FLVN01,Metalúrgica Alvorada Eireli,ALVORADA,BR1,16938471000195,Rua Conceição,612,13.010-050,Campinas,SP,BR,1410,2006,1,12119000,2006,0001,237,99887-1,BRL
9,021,0001,FLVN00,FLVN01,Química Progresso Me,PROGRESSO,BR1,31845962000119,"Rua Frei Gaspar, 2140",<NA>,09720-440,Sao Bernardo do Campo,SP,BR,1410,2006,0001,12119000,2006,1,33,77441-2,BRL


### 2: Ajustes Finos (Siglas Societárias e Acentos nas Cidades)

In [11]:
# 1. Manter siglas societárias em MAIÚSCULO na Razão Social
siglas = {
    r"\bLtda\.?\b": "LTDA",
    r"\bEpp\.?\b": "EPP",
    r"\bMe\.?\b": "ME",
    r"\bEireli\.?\b": "EIRELI",
    r"\bS\.?A\.?\b": "S.A.",
}
for errado, certo in siglas.items():
    df["NAME_ORG1 (Razao Social)"] = df["NAME_ORG1 (Razao Social)"].str.replace(
        errado, certo, regex=True
    )

# 2. Corrigir "Dom Pedro Ii" -> "Dom Pedro II" nas ruas
df["STREET (Rua)"] = df["STREET (Rua)"].str.replace(r"\bIi\b", "II", regex=True)

# 3. Corrigir acentuação das cidades de SP
cidades_corretas = {
    "Sao Paulo": "São Paulo",
    "Sao Bernardo do Campo": "São Bernardo do Campo",
    "Santo Andre": "Santo André",
    "Jundiai": "Jundiaí",
    "Ribeirao Preto": "Ribeirão Preto",
    "Sao Jose dos Campos": "São José dos Campos",
}
df["CITY1 (Cidade SP)"] = df["CITY1 (Cidade SP)"].replace(cidades_corretas)

display(df[["NAME_ORG1 (Razao Social)", "STREET (Rua)", "CITY1 (Cidade SP)"]].head(10))


,NAME_ORG1 (Razao Social),STREET (Rua),CITY1 (Cidade SP)
0,Metalúrgica Sol Nascente LTDA,Rua Jurubatuba,São Bernardo do Campo
1,Transportadora Meridional LTDA,"Avenida Paulista, 1578",São Paulo
2,Distribuidora Aurora EIRELI,Rua Sete de Setembro,Guarulhos
3,Ferramentas Bela Vista EPP,"Rua Voluntários da Pátria, 3184",São Paulo
4,Embalagens Cambuí Sul LTDA,Avenida Moraes Salles,Campinas
5,Plásticos Sorocabana ME,"Rua Quinze de Novembro, 2740",Sorocaba
6,Química Serra Azul EPP,Avenida Portugal,Santo André
7,Metalúrgica Paraíba LTDA.,"Avenida Nelson D'Ávila, 820",São José dos Campos
8,Metalúrgica Alvorada EIRELI,Rua Conceição,Campinas
9,Química Progresso ME,"Rua Frei Gaspar, 2140",São Bernardo do Campo


* #### Limpar espaços extras e padronizar letras (Maiúsculas / Minúsculas)

In [12]:
# 1. Tirar espaços nas pontas e espaços duplos no meio de TODAS as colunas
for col in df.columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    ) # type: ignore

# 2. Manter código do fornecedor no padrão original de 3 dígitos ('016', '036', etc.)
df["PARTNER (Codigo BP)"] = (
    df["PARTNER (Codigo BP)"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(3)
)


# 3. Colunas que devem ficar 100% em MAIÚSCULO
cols_maiusculas = [
    "AGRUPAMENTO",
    "FUNCAO_1",
    "FUNCAO_2",
    "BU_SORT1 (Termo Pesquisa)",
    "TAXTYPE",
    "REGION (UF)",
    "COUNTRY",
    "LGORT (Deposito)",
    "WAERS (Moeda)",
]
for col in cols_maiusculas:
    df[col] = df[col].str.upper()

# 4. Colunas de texto em Título (Primeira letra maiúscula + preposições minúsculas)
preposicoes = [" Da ", " De ", " Do ", " Das ", " Dos ", " E "]
cols_titulo = ["NAME_ORG1 (Razao Social)", "STREET (Rua)", "CITY1 (Cidade SP)"]

for col in cols_titulo:
    df[col] = df[col].str.title()
    for prep in preposicoes:
        df[col] = df[col].str.replace(prep, prep.lower(), regex=False)

# 5. Limpar CNPJ (remove '/', '-', '.', espaços e mantém os 14 dígitos numéricos)
df["TAXNUM (CNPJ)"] = (
    df["TAXNUM (CNPJ)"].str.replace(r"\D", "", regex=True).str.zfill(14)
)

display(df.head(10))


,PARTNER (Codigo BP),AGRUPAMENTO,FUNCAO_1,FUNCAO_2,NAME_ORG1 (Razao Social),BU_SORT1 (Termo Pesquisa),TAXTYPE,TAXNUM (CNPJ),STREET (Rua),HOUSE_NUM1 (Numero),POST_CODE1 (CEP SP),CITY1 (Cidade SP),REGION (UF),COUNTRY,BUKRS (Empresa),EKORG (Org Compras),ZTERM (Cond Pagamento),AKONT (Conta Conciliacao),KOSTL (Centro de Custo),LGORT (Deposito),BANKL (Chave Banco),BANKN (Conta Bancaria),WAERS (Moeda)
0,016,0001,FLVN00,FLVN01,Metalúrgica Sol Nascente Ltda,SOL NASCEN,BR1,26819472000131,Rua Jurubatuba,845,09.725-000,São Bernardo do Campo,SP,BR,1410,2006,0001,12119000,1410,141A,341,11223-4,BRL
1,036,0001,FLVN00,FLVN01,Transportadora Meridional Ltda,MERIDIONAL,BR1,46819247000163,"Avenida Paulista, 1578",<NA>,01310-200,São Paulo,SP,BR,1410,2006,0001,12119000,1410,1,104,33210-8,BRL
2,010,0001,FLVN00,FLVN01,Distribuidora Aurora Eireli,AURORA,BR1,20846193000174,Rua Sete de Setembro,2890,07011-020,Guarulhos,SP,BR,1410,2006,1,12119000,2006,0001,237,99887-1,BRL
3,015,0001,FLVN00,FLVN01,Ferramentas Bela Vista Epp,BELA VISTA,BR1,25891437000128,"Rua Voluntários da Pátria, 3184",<NA>,02011000,São Paulo,SP,BR,1410,1410,0001,12119000,1410,1,237,99887-1,BRL
4,032,0001,FLVN00,FLVN01,Embalagens Cambuí Sul Ltda,CAMBUÍ SUL,BR1,42891634000182,Avenida Moraes Salles,1730,13.010-002,Campinas,SP,BR,1410,1410,0001,12119000,2006,141A,341,11223-4,BRL
5,025,0001,FLVN00,FLVN01,Plásticos Sorocabana Me,SOROCABANA,BR1,35841926000172,"Rua Quinze de Novembro, 2740",<NA>,18010-080,Sorocaba,SP,BR,1410,1410,1,12119000,1410,1,1,56789-0,BRL
6,003,0001,FLVN00,FLVN01,Química Serra Azul Epp,SERRA AZUL,BR1,13619482000153,Avenida Portugal,945,09040-000,Santo André,SP,BR,1410,2006,0001,12119000.0,2006,0001,341,11223-4,BRL
7,027,0001,FLVN00,FLVN01,Metalúrgica Paraíba Ltda.,PARAÍBA,BR1,37849152000168,"Avenida Nelson D'Ávila, 820",<NA>,12245030,São José dos Campos,SP,BR,1410,2006,0001,12119000,1410,141A,104,33210-8,BRL
8,006,0001,FLVN00,FLVN01,Metalúrgica Alvorada Eireli,ALVORADA,BR1,16938471000195,Rua Conceição,612,13.010-050,Campinas,SP,BR,1410,2006,1,12119000,2006,0001,237,99887-1,BRL
9,021,0001,FLVN00,FLVN01,Química Progresso Me,PROGRESSO,BR1,31845962000119,"Rua Frei Gaspar, 2140",<NA>,09720-440,São Bernardo do Campo,SP,BR,1410,2006,0001,12119000,2006,1,33,77441-2,BRL


 ### 3: Extrair Número da Rua (com ou sem vírgula) e Limpar <NA>


In [13]:
# 1. Substituir qualquer texto '<NA>' por nulo real antes de extrair
df["HOUSE_NUM1 (Numero)"] = df["HOUSE_NUM1 (Numero)"].replace({"<NA>": pd.NA, "nan": pd.NA, "": pd.NA})

# 2. Extrair o número do final da coluna STREET (aceita com vírgula 'Rua X, 123' OU sem vírgula 'Rua X 123')
sem_numero = df["HOUSE_NUM1 (Numero)"].isna()
extraido = df.loc[sem_numero, "STREET (Rua)"].str.extract(r"^(.*?)[,\s]+(\d+)$")

# Atualiza apenas onde conseguiu extrair um número válido
achou_numero = extraido[1].notna()
indices_atualizar = extraido[achou_numero].index

df.loc[indices_atualizar, "STREET (Rua)"] = extraido.loc[achou_numero, 0].str.strip()
df.loc[indices_atualizar, "HOUSE_NUM1 (Numero)"] = extraido.loc[achou_numero, 1].str.strip()

# Limpa qualquer decimal fantasma (.0) e garante string vazia onde realmente não existe número
df["HOUSE_NUM1 (Numero)"] = (
    df["HOUSE_NUM1 (Numero)"]
    .fillna("")
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .replace({"<NA>": "", "nan": "", "None": ""})
)

# 3. Formatar CEP com hífen (9 caracteres: 00000-000) exigido pelo SAP Brasil
df["POST_CODE1 (CEP SP)"] = (
    df["POST_CODE1 (CEP SP)"]
    .str.replace(r"\D", "", regex=True)
    .str.zfill(8)
    .str.replace(r"^(\d{5})(\d{3})$", r"\1-\2", regex=True)
)

# 4. Corrigir zeros à esquerda e decimais fantasmas (.0) das colunas SAP/Banco
df["AKONT (Conta Conciliacao)"] = df["AKONT (Conta Conciliacao)"].str.replace(r"\.0$", "", regex=True)
df["ZTERM (Cond Pagamento)"] = df["ZTERM (Cond Pagamento)"].str.replace(r"\.0$", "", regex=True).str.zfill(4)
df["BANKL (Chave Banco)"] = df["BANKL (Chave Banco)"].str.replace(r"\.0$", "", regex=True).str.zfill(3)

if "LGORT (Deposito)" in df.columns:
    df["LGORT (Deposito)"] = df["LGORT (Deposito)"].apply(
        lambda x: x.zfill(4) if pd.notna(x) and str(x).isdigit() else x
    )

display(df[["PARTNER (Codigo BP)", "STREET (Rua)", "HOUSE_NUM1 (Numero)", "POST_CODE1 (CEP SP)"]].head(10))


,PARTNER (Codigo BP),STREET (Rua),HOUSE_NUM1 (Numero),POST_CODE1 (CEP SP)
0,016,Rua Jurubatuba,845,09725-000
1,036,Avenida Paulista,1578,01310-200
2,010,Rua Sete de Setembro,2890,07011-020
3,015,Rua Voluntários da Pátria,3184,02011-000
4,032,Avenida Moraes Salles,1730,13010-002
5,025,Rua Quinze de Novembro,2740,18010-080
6,003,Avenida Portugal,945,09040-000
7,027,Avenida Nelson D'Ávila,820,12245-030
8,006,Rua Conceição,612,13010-050
9,021,Rua Frei Gaspar,2140,09720-440


### 4: Auditar, Remover Duplicatas e Substituir por CNPJs Válidos

In [14]:
# 1. Conferir quantos CNPJs duplicados existem antes de apagar
duplicados = df[df.duplicated(subset=["TAXNUM (CNPJ)"], keep=False)].sort_values(
    by="PARTNER (Codigo BP)"
)
print(f"Total de linhas duplicadas encontradas (pares): {len(duplicados)}")

# 2. Remover duplicatas mantendo o primeiro cadastro e ordenar do '001' ao '040'
df = (
    df.drop_duplicates(subset=["TAXNUM (CNPJ)"], keep="first")
    .sort_values(by="PARTNER (Codigo BP)")
    .reset_index(drop=True)
)

# 3. Gerar 40 novos CNPJs fictícios inéditos e 100% válidos (usando nova semente = 500)
cnpjs_validos = set()
semente_atual = 500
while len(cnpjs_validos) < len(df):
    cnpjs_validos.add(gerar_cnpj_valido(semente_atual))
    semente_atual += 1

df["TAXNUM (CNPJ)"] = sorted(list(cnpjs_validos))

print(f"Total final de fornecedores únicos com CNPJs inéditos: {len(df)}")
display(
    df[
        [
            "PARTNER (Codigo BP)",
            "AGRUPAMENTO",
            "NAME_ORG1 (Razao Social)",
            "TAXNUM (CNPJ)",
        ]
    ].head(10)
)


Total de linhas duplicadas encontradas (pares): 20
Total final de fornecedores únicos com CNPJs inéditos: 40


,PARTNER (Codigo BP),AGRUPAMENTO,NAME_ORG1 (Razao Social),TAXNUM (CNPJ)
0,001,0001,Comércio Atlântico Ltda.,00662766000131
1,002,0001,Plásticos Horizonte Ltda,03292694000176
2,003,0001,Química Serra Azul Epp,07256596000134
3,004,0001,Metalúrgica Vale Verde Eireli,09417792000179
4,005,0001,Química Planalto Epp,10446255000136
5,006,0001,Metalúrgica Alvorada Eireli,13690218000167
6,007,0001,Distribuidora Boa Vista Eireli,17123734000160
7,008,0001,Química Monte Alegre Me,22536420000120
8,009,0001,Logística Primavera Me,23204667000101
9,010,0001,Distribuidora Aurora Eireli,23736366000129


### Auditar e Remover Duplicatas 

#### (Mostra os registros repetidos antes de excluir e salva o Excel final)

In [15]:
# 1. Conferir quantos CNPJs duplicados existem antes de apagar
duplicados = df[df.duplicated(subset=["TAXNUM (CNPJ)"], keep=False)].sort_values(
    by="PARTNER (Codigo BP)"
)
print(f"Total de linhas duplicadas encontradas: {len(duplicados)}")
display(
    duplicados[
        [
            "PARTNER (Codigo BP)",
            "NAME_ORG1 (Razao Social)",
            "TAXNUM (CNPJ)",
            "EKORG (Org Compras)",
            "KOSTL (Centro de Custo)",
        ]
    ]
)

# 2. Remover duplicatas mantendo o primeiro cadastro e ordenar pelo Código BP
df = (
    df.drop_duplicates(subset=["TAXNUM (CNPJ)"], keep="first")
    .sort_values(by="PARTNER (Codigo BP)")
    .reset_index(drop=True)
)

print(f"Total final de fornecedores únicos: {len(df)}")
display(df.head(10))

Total de linhas duplicadas encontradas: 0


,PARTNER (Codigo BP),NAME_ORG1 (Razao Social),TAXNUM (CNPJ),EKORG (Org Compras),KOSTL (Centro de Custo)


Total final de fornecedores únicos: 40


,PARTNER (Codigo BP),AGRUPAMENTO,FUNCAO_1,FUNCAO_2,NAME_ORG1 (Razao Social),BU_SORT1 (Termo Pesquisa),TAXTYPE,TAXNUM (CNPJ),STREET (Rua),HOUSE_NUM1 (Numero),POST_CODE1 (CEP SP),CITY1 (Cidade SP),REGION (UF),COUNTRY,BUKRS (Empresa),EKORG (Org Compras),ZTERM (Cond Pagamento),AKONT (Conta Conciliacao),KOSTL (Centro de Custo),LGORT (Deposito),BANKL (Chave Banco),BANKN (Conta Bancaria),WAERS (Moeda)
0,001,0001,FLVN00,FLVN01,Comércio Atlântico Ltda.,ATLÂNTICO,BR1,00662766000131,Rua Consolação,1420,13400-120,Piracicaba,SP,BR,1410,1410,0001,12119000,1410,141A,104,33210-8,BRL
1,002,0001,FLVN00,FLVN01,Plásticos Horizonte Ltda,HORIZONTE,BR1,03292694000176,Rua Oscar Freire,2310,01426-001,São Paulo,SP,BR,1410,1410,0001,12119000,1410,0001,341,11223-4,BRL
2,003,0001,FLVN00,FLVN01,Química Serra Azul Epp,SERRA AZUL,BR1,07256596000134,Avenida Portugal,945,09040-000,Santo André,SP,BR,1410,2006,0001,12119000,2006,0001,341,11223-4,BRL
3,004,0001,FLVN00,FLVN01,Metalúrgica Vale Verde Eireli,VALE VERDE,BR1,09417792000179,Avenida Francisco Glicério,1180,13012-100,Campinas,SP,BR,1410,2006,0001,12119000,2006,0001,001,56789-0,BRL
4,005,0001,FLVN00,FLVN01,Química Planalto Epp,PLANALTO,BR1,10446255000136,Rua Barão de Itapura,1890,13020-432,Campinas,SP,BR,1410,2006,0001,12119000,1410,141A,001,56789-0,BRL
5,006,0001,FLVN00,FLVN01,Metalúrgica Alvorada Eireli,ALVORADA,BR1,13690218000167,Rua Conceição,612,13010-050,Campinas,SP,BR,1410,2006,0001,12119000,2006,0001,237,99887-1,BRL
6,007,0001,FLVN00,FLVN01,Distribuidora Boa Vista Eireli,BOA VISTA,BR1,17123734000160,Avenida dos Autonomistas,3150,06020-010,Osasco,SP,BR,1410,1410,0001,12119000,1410,141A,341,11223-4,BRL
7,008,0001,FLVN00,FLVN01,Química Monte Alegre Me,MONTE ALEG,BR1,22536420000120,Rua Coronel Oliveira Lima,740,09010-000,Santo André,SP,BR,1410,2006,0001,12119000,1410,141A,001,56789-0,BRL
8,009,0001,FLVN00,FLVN01,Logística Primavera Me,PRIMAVERA,BR1,23204667000101,Rua Senador Flaquer,1530,09010-160,Santo André,SP,BR,1410,1410,0001,12119000,2006,0001,033,77441-2,BRL
9,010,0001,FLVN00,FLVN01,Distribuidora Aurora Eireli,AURORA,BR1,23736366000129,Rua Sete de Setembro,2890,07011-020,Guarulhos,SP,BR,1410,2006,0001,12119000,2006,0001,237,99887-1,BRL


* ### Remover <NA> no Excel

In [16]:
# Limpa qualquer valor nulo/pd.NA para string vazia antes de salvar no Excel
df = df.fillna("").replace({"<NA>": "", "nan": "", "None": ""})
print(f"Total final de fornecedores únicos: {len(df)}")
display(df.head(10))

Total final de fornecedores únicos: 40


,PARTNER (Codigo BP),AGRUPAMENTO,FUNCAO_1,FUNCAO_2,NAME_ORG1 (Razao Social),BU_SORT1 (Termo Pesquisa),TAXTYPE,TAXNUM (CNPJ),STREET (Rua),HOUSE_NUM1 (Numero),POST_CODE1 (CEP SP),CITY1 (Cidade SP),REGION (UF),COUNTRY,BUKRS (Empresa),EKORG (Org Compras),ZTERM (Cond Pagamento),AKONT (Conta Conciliacao),KOSTL (Centro de Custo),LGORT (Deposito),BANKL (Chave Banco),BANKN (Conta Bancaria),WAERS (Moeda)
0,001,0001,FLVN00,FLVN01,Comércio Atlântico Ltda.,ATLÂNTICO,BR1,00662766000131,Rua Consolação,1420,13400-120,Piracicaba,SP,BR,1410,1410,0001,12119000,1410,141A,104,33210-8,BRL
1,002,0001,FLVN00,FLVN01,Plásticos Horizonte Ltda,HORIZONTE,BR1,03292694000176,Rua Oscar Freire,2310,01426-001,São Paulo,SP,BR,1410,1410,0001,12119000,1410,0001,341,11223-4,BRL
2,003,0001,FLVN00,FLVN01,Química Serra Azul Epp,SERRA AZUL,BR1,07256596000134,Avenida Portugal,945,09040-000,Santo André,SP,BR,1410,2006,0001,12119000,2006,0001,341,11223-4,BRL
3,004,0001,FLVN00,FLVN01,Metalúrgica Vale Verde Eireli,VALE VERDE,BR1,09417792000179,Avenida Francisco Glicério,1180,13012-100,Campinas,SP,BR,1410,2006,0001,12119000,2006,0001,001,56789-0,BRL
4,005,0001,FLVN00,FLVN01,Química Planalto Epp,PLANALTO,BR1,10446255000136,Rua Barão de Itapura,1890,13020-432,Campinas,SP,BR,1410,2006,0001,12119000,1410,141A,001,56789-0,BRL
5,006,0001,FLVN00,FLVN01,Metalúrgica Alvorada Eireli,ALVORADA,BR1,13690218000167,Rua Conceição,612,13010-050,Campinas,SP,BR,1410,2006,0001,12119000,2006,0001,237,99887-1,BRL
6,007,0001,FLVN00,FLVN01,Distribuidora Boa Vista Eireli,BOA VISTA,BR1,17123734000160,Avenida dos Autonomistas,3150,06020-010,Osasco,SP,BR,1410,1410,0001,12119000,1410,141A,341,11223-4,BRL
7,008,0001,FLVN00,FLVN01,Química Monte Alegre Me,MONTE ALEG,BR1,22536420000120,Rua Coronel Oliveira Lima,740,09010-000,Santo André,SP,BR,1410,2006,0001,12119000,1410,141A,001,56789-0,BRL
8,009,0001,FLVN00,FLVN01,Logística Primavera Me,PRIMAVERA,BR1,23204667000101,Rua Senador Flaquer,1530,09010-160,Santo André,SP,BR,1410,1410,0001,12119000,2006,0001,033,77441-2,BRL
9,010,0001,FLVN00,FLVN01,Distribuidora Aurora Eireli,AURORA,BR1,23736366000129,Rua Sete de Setembro,2890,07011-020,Guarulhos,SP,BR,1410,2006,0001,12119000,2006,0001,237,99887-1,BRL


### 5: Ajuste Final de Bancos/Razão Social e Exportação na Pasta Atual

In [17]:
# 1. Corrigir 'S.A..' -> 'S.A.' e remover ponto final sobrando em 'LTDA.', 'EPP.', 'ME.', 'EIRELI.'
df["NAME_ORG1 (Razao Social)"] = (
    df["NAME_ORG1 (Razao Social)"]
    .str.replace(r"S\.A\.\.+", "S.A.", regex=True)
    .str.replace(r"\b(LTDA|EPP|ME|EIRELI)\.+$", r"\1", regex=True)
)

# Corrigir numeral romano 'Dom Pedro Ii' -> 'Dom Pedro II' nas ruas
df["STREET (Rua)"] = df["STREET (Rua)"].str.replace(r"\bIi\b", "II", regex=True)

# 2. Padronizar Chaves de Banco (BANKL) conforme a tabela BNKA do seu SAP (máximo 8 dígitos!)
mapa_bancos_sap = {
    "341": "34192978",  # Banco Itau (8 dígitos)
    "237": "23721234",  # Bank 2 - SAMPLE BANK (8 dígitos)
    "104": "34171234",  # Bank 1 - SAMPLE BANK (8 dígitos)
    "001": "23721234",  # Banco 2 (corrigido de 9 para 8 dígitos)
    "033": "341",       # Bank 3 - SAMPLE BANK
}
df["BANKL (Chave Banco)"] = (
    df["BANKL (Chave Banco)"]
    .astype(str)
    .str.strip()
    .replace(mapa_bancos_sap)
    .replace({"237212345": "23721234"})
)

# 3. Separar Conta e Dígito (caso tenha hífen) OU recuperar o dígito exato da conta
mapa_digitos = {"33210": "8", "11223": "4", "56789": "0", "99887": "1", "77441": "2"}

if df["BANKN (Conta Bancaria)"].astype(str).str.contains("-").any():
    conta_split = (
        df["BANKN (Conta Bancaria)"]
        .astype(str)
        .str.strip()
        .str.split("-", n=1, expand=True)
    )
    df["BANKN (Conta Bancaria)"] = conta_split[0].str.strip()
    bkont_valores = conta_split[1].fillna("").str.strip()
else:
    df["BANKN (Conta Bancaria)"] = df["BANKN (Conta Bancaria)"].astype(str).str.strip()
    bkont_valores = df["BANKN (Conta Bancaria)"].map(mapa_digitos).fillna("")

df = df.drop(columns=["BKONT (Chave Controle)"], errors="ignore")
pos_bankn = df.columns.get_loc("BANKN (Conta Bancaria)") + 1
df.insert(pos_bankn, "BKONT (Chave Controle)", bkont_valores)

# 4. Garantir que WAERS (Moeda) fique antes de BANKL (Chave Banco)
col_moeda = df.pop("WAERS (Moeda)")
pos_bankl = df.columns.get_loc("BANKL (Chave Banco)")
df.insert(pos_bankl, "WAERS (Moeda)", col_moeda)

# Garante que o Agrupamento (BU_GROUP) nunca fique em branco (campo obrigatório no SAP)
df["AGRUPAMENTO"] = (
    df["AGRUPAMENTO"]
    .fillna("")
    .astype(str)
    .str.strip()
    .replace({"": "0001", "<NA>": "0001", "nan": "0001", "None": "0001"})
    .str.zfill(4)
)

# Limpa qualquer resquício de '<NA>' ou 'nan' antes de exportar para o Excel
df = df.fillna("").replace({"<NA>": "", "nan": "", "None": ""})

# 5. Salvar diretamente na pasta atual (BASE_DADOS_LIMPA)
caminho_arquivo = "Base_Limpa_BP_Fornecedor.xlsx"
df.to_excel(caminho_arquivo, index=False)

print(f"✅ Arquivo '{caminho_arquivo}' salvo com sucesso ({len(df)} registros)!")
display(
    df[
        [
            "PARTNER (Codigo BP)",
            "AGRUPAMENTO",
            "TAXNUM (CNPJ)",
            "POST_CODE1 (CEP SP)",
            "BANKL (Chave Banco)",
            "BANKN (Conta Bancaria)",
            "BKONT (Chave Controle)",
        ]
    ].head(10)
)


✅ Arquivo 'Base_Limpa_BP_Fornecedor.xlsx' salvo com sucesso (40 registros)!


,PARTNER (Codigo BP),AGRUPAMENTO,TAXNUM (CNPJ),POST_CODE1 (CEP SP),BANKL (Chave Banco),BANKN (Conta Bancaria),BKONT (Chave Controle)
0,001,0001,00662766000131,13400-120,34171234,33210,8
1,002,0001,03292694000176,01426-001,34192978,11223,4
2,003,0001,07256596000134,09040-000,34192978,11223,4
3,004,0001,09417792000179,13012-100,23721234,56789,0
4,005,0001,10446255000136,13020-432,23721234,56789,0
5,006,0001,13690218000167,13010-050,23721234,99887,1
6,007,0001,17123734000160,06020-010,34192978,11223,4
7,008,0001,22536420000120,09010-000,23721234,56789,0
8,009,0001,23204667000101,09010-160,341,77441,2
9,010,0001,23736366000129,07011-020,23721234,99887,1


#### 6: Auditoria do XML na Pasta AUTOMACAO_LTMC

In [18]:
caminho_xml = "../AUTOMACAO_LTMC/PT_Fornecedor_LTMC_Preenchidos.xml"

# 1. Verifica o cabeçalho exigido pelo SAP LTMC
with open(caminho_xml, "r", encoding="utf-8") as f:
    primeiras_linhas = [next(f).strip() for _ in range(2)]

print("🔍 1. CABEÇALHO XML SAP:")
for linha in primeiras_linhas:
    print(f"   {linha}")

# 2. Analisa todas as abas existentes no XML automaticamente
tree = ET.parse(caminho_xml)
root = tree.getroot()
NS_SS = "urn:schemas-microsoft-com:office:spreadsheet"
ns = {"ss": NS_SS}

print("\n📊 2. RESUMO DE TODAS AS ABAS NO XML:")
for ws in root.findall("ss:Worksheet", ns):
    nome_aba = ws.attrib.get(f"{{{NS_SS}}}Name")
    table = ws.find("ss:Table", ns)
    rows = table.findall("ss:Row", ns)
    qtd_dados = max(0, len(rows) - 8)  # Desconta as 8 linhas de cabeçalho do SAP

    amostra = []
    if qtd_dados > 0:
        for cell in rows[8].findall("ss:Cell", ns):
            data = cell.find("ss:Data", ns)
            if data is not None and data.text:
                amostra.append(data.text)

    status = "✅" if qtd_dados > 0 else "⚠️ (Vazia)"
    print(f"   {status} {repr(nome_aba)}: {qtd_dados} registros | Amostra: {amostra}")


🔍 1. CABEÇALHO XML SAP:
   <?xml version="1.0"?>
   <?mso-application progid="Excel.Sheet"?>

📊 2. RESUMO DE TODAS AS ABAS NO XML:
   ✅ 'Introdução': 75 registros | Amostra: [' ']
   ✅ 'Lista de campos': 512 registros | Amostra: ['Organização do fornecedor', 'Nome', 'Texto', '40    ', 'S_SUPPL_GEN', 'NAME_FIRST']
   ✅ 'Dados gerais': 40 registros | Amostra: ['900001', '0001', 'Comércio Atlântico Ltda.', 'ATLÂNTICO', 'Rua Consolação', '1420', '13400-120', 'Piracicaba', 'BR', 'SP', 'PT']
   ✅ 'Funções do parceiro de negócio': 80 registros | Amostra: ['900001', 'FLVN00']
   ⚠️ (Vazia) 'Endereços adicionais': 0 registros | Amostra: []
   ⚠️ (Vazia) 'Textos gerais': 0 registros | Amostra: []
   ✅ 'Dados da empresa': 40 registros | Amostra: ['900001', '1410', '12119000', '0001']
   ⚠️ (Vazia) 'Dados do imposto retido na fon': 0 registros | Amostra: []
   ⚠️ (Vazia) 'Textos da empresa': 0 registros | Amostra: []
   ✅ 'Dados de organização de compra': 40 registros | Amostra: ['900001', '1410',